In [4]:
from langchain.chat_models import init_chat_model
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory


# =========================================================
# 1. Model
# =========================================================

model = init_chat_model(
    "qwen3:1.7b",
    model_provider="ollama",
    temperature=0,
)


# =========================================================
# 2. Prompt
# =========================================================

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant."
    ),

    # اینجا History قرار می‌گیرد
    MessagesPlaceholder(variable_name="history"),

    # سؤال فعلی
    ("human", "{question}"),
])


# =========================================================
# 3. Chain
# =========================================================

chain = prompt | model


# =========================================================
# 4. Storage برای History
# =========================================================

store = {}


def get_session_history(session_id: str):

    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()

    return store[session_id]


# =========================================================
# 5. اضافه کردن قابلیت Message History
# =========================================================

conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,

    # نام فیلد سؤال در input
    input_messages_key="question",

    # نام Placeholder در Prompt
    history_messages_key="history",
)


# =========================================================
# 6. سؤال اول
# =========================================================

response = conversation.invoke(
    {
        "question": "My name is Donald"
    },
    config={
        "configurable": {
            "session_id": "user_125"
        }
    }
)

print("AI:", response.content)


# =========================================================
# 7. سؤال دوم
# =========================================================

response = conversation.invoke(
    {
        "question": "What is my name?"
    },
    config={
        "configurable": {
            "session_id": "user_125"
        }
    }
)

print("AI:", response.content)

d:\Projects\llm_engineering\llm_engineering\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


AI: Hello, Donald! How can I assist you today? 😊
AI: Your name is Donald. Let me know if there's anything I can assist you with! 😊
